# Figure 6 Reproducibility: `uncertainty_estimator` Threshold Calibration

This notebook reproduces the final Figure 6 threshold-calibration workflow:
1. Start from `models/uncertainty_estimator` (trained in Figure 5; no retraining here).
2. Recalibrate only the uncertainty threshold using semantic+proteomics dev predictions:
   - `predictions/Uncertainty_experiment_semantic_proteomics/binary_loo_ablation/semantic_BioBERT_tabm_Text_Embedding+Proteomics`
   - `predictions/Uncertainty_experiment_semantic_proteomics/binary_loso_ablation/semantic_BioBERT_tabm_Text_Embedding+Proteomics`
3. Export the adapted model to `models/uncertainty_estimator_adapted`.
4. Apply it to `predictions/external_data/sig_reduced/` and write outputs to
   `predictions/external_data/sig_reduced/uncertainty_meta_apply/`.
5. Calibration uses only LOO/LOSO dev data; no external test labels are used.


In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = None
for parent in [cwd, *cwd.parents]:
    if (parent / "Data").exists() and (parent / "predictions").exists():
        REPO_ROOT = parent
        break
if REPO_ROOT is None:
    REPO_ROOT = cwd
os.chdir(REPO_ROOT)

src_candidates = [
    REPO_ROOT / "scripts",
    REPO_ROOT / "notebooks",
    REPO_ROOT / "code_repository" / "scripts",
    REPO_ROOT / "code_repository" / "notebooks",
]
for src in reversed(src_candidates):
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))

print(f"Working dir: {Path.cwd()}")


Working dir: /mnt/nas/PhD/Code/Ms_paint/code_repository


In [2]:
import json
import re
import shutil

import numpy as np
import pandas as pd

from uncertainty.uncertainty_estimator import UncertaintyEstimator

BASE_META_DIR = Path("models/uncertainty_estimator")
RECAL_META_DIR = Path("models/uncertainty_estimator_adapted")

LOO_ROOT = Path(
    "predictions/Uncertainty_experiment_semantic_proteomics/binary_loo_ablation/"
    "semantic_BioBERT_tabm_Text_Embedding+Proteomics"
)
LOSO_ROOT = Path(
    "predictions/Uncertainty_experiment_semantic_proteomics/binary_loso_ablation/"
    "semantic_BioBERT_tabm_Text_Embedding+Proteomics"
)

EXTERNAL_SIG_ROOT = Path("predictions/external_data/sig_reduced")
EXTERNAL_OUT_DIR = Path("predictions/external_data/sig_reduced/uncertainty_meta_apply")

ABS_MIN_SCORE_FLOOR = 0.05
SCORE_FLOOR_FACTOR = 0.1
MARGIN_FLOOR = 0.01

for p in [BASE_META_DIR, LOO_ROOT, LOSO_ROOT, EXTERNAL_SIG_ROOT]:
    if not p.exists():
        raise FileNotFoundError(p)

base_artifact = json.loads((BASE_META_DIR / "meta_artifact.json").read_text())
ALPHA = float(base_artifact.get("target_id_fpr", 0.1))

print("Base meta dir:", BASE_META_DIR)
print("Recal meta dir:", RECAL_META_DIR)
print("LOO root:", LOO_ROOT)
print("LOSO root:", LOSO_ROOT)
print("External sig root:", EXTERNAL_SIG_ROOT)
print("Target ID-FPR (alpha):", ALPHA)



Base meta dir: models/uncertainty_estimator
Recal meta dir: models/uncertainty_estimator_adapted
LOO root: predictions/Uncertainty_experiment_semantic_proteomics/binary_loo_ablation/semantic_BioBERT_tabm_Text_Embedding+Proteomics
LOSO root: predictions/Uncertainty_experiment_semantic_proteomics/binary_loso_ablation/semantic_BioBERT_tabm_Text_Embedding+Proteomics
External sig root: predictions/external_data/sig_reduced
Target ID-FPR (alpha): 0.1


In [3]:
def _extract_seed_from_name(stem):
    m = re.search(r"seed(\d+)", stem)
    return int(m.group(1)) if m else -1


def ensure_pairwise_csvs(run_root, force_rebuild=False):
    pair_csv = run_root / "pairwise_predictions.csv"
    pair_seed_csv = run_root / "pairwise_predictions_by_seed.csv"

    required_pair_cols = {"SampleID", "CompoundID", "CandidateMoA", "Aug_Index", "Score"}
    required_seed_cols = {"RowID", "SampleID", "CompoundID", "CandidateMoA", "Aug_Index", "Seed", "Score"}

    if (not force_rebuild) and pair_csv.exists() and pair_seed_csv.exists():
        ok = True
        try:
            c1 = set(pd.read_csv(pair_csv, nrows=1).columns)
            c2 = set(pd.read_csv(pair_seed_csv, nrows=1).columns)
            ok = required_pair_cols.issubset(c1) and required_seed_cols.issubset(c2)
        except Exception:
            ok = False
        if ok:
            print(f"Using existing pairwise CSVs in {run_root}")
            return pair_csv, pair_seed_csv

    pkl_files = sorted(run_root.glob("fold*_rep*_seed*.pkl"))
    if not pkl_files:
        raise FileNotFoundError(f"No fold*_rep*_seed*.pkl files found in {run_root}")

    all_rows = []
    for pkl_path in pkl_files:
        obj = pd.read_pickle(pkl_path)
        if not isinstance(obj, pd.DataFrame):
            continue
        df = obj.copy()

        local_rowid = df["RowID"].astype(str).values if "RowID" in df.columns else df.index.astype(str)
        needed = {"SampleID", "CompoundID", "CandidateMoA", "ProbaPos"}
        if not needed.issubset(df.columns):
            continue

        seed = int(df["seed"].iloc[0]) if "seed" in df.columns else _extract_seed_from_name(pkl_path.stem)
        fold_tag = pkl_path.stem

        out = pd.DataFrame(
            {
                "RowID": [f"{fold_tag}__{rid}" for rid in local_rowid],
                "SampleID": df["SampleID"].astype(str).values,
                "CompoundID": df["CompoundID"].astype(str).values,
                "CandidateMoA": df["CandidateMoA"].astype(str).values,
                "Aug_Index": np.zeros(len(df), dtype=int),
                "Seed": int(seed),
                "Score": pd.to_numeric(df["ProbaPos"], errors="coerce").astype(float).values,
            }
        )
        all_rows.append(out)

    if not all_rows:
        raise RuntimeError(f"Could not build pairwise rows from PKLs in {run_root}")

    by_seed = pd.concat(all_rows, ignore_index=True)
    by_seed = by_seed.dropna(subset=["Score"]).reset_index(drop=True)

    pairwise = (
        by_seed
        .groupby(["SampleID", "CompoundID", "CandidateMoA", "Aug_Index"], as_index=False)["Score"]
        .mean()
    )

    by_seed.to_csv(pair_seed_csv, index=False)
    pairwise.to_csv(pair_csv, index=False)

    print(f"Rebuilt: {pair_csv}")
    print(f"Rebuilt: {pair_seed_csv}")
    return pair_csv, pair_seed_csv


def load_true_moa_by_compound():
    y = pd.read_pickle(Path("Data/Base/y_labels.pkl"))
    y = pd.Series(y) if isinstance(y, dict) else y
    y.index = y.index.astype(str)
    comp = y.index.map(lambda s: s.split("_")[0])
    out = y.groupby(comp).first().astype(str)
    out.index.name = "CompoundID"
    return out


def evaluate_dev_root(run_root, meta_dir, dataset_name, true_moa_by_compound):
    ensure_pairwise_csvs(run_root)

    ue = UncertaintyEstimator(
        pred_root=str(run_root),
        meta_dir=str(meta_dir),
        out_dir=str(run_root / "uncertainty_meta_apply_recalibration_dev"),
        output_prefix=f"dev_{dataset_name.lower()}",
        abs_min_score_floor=0.0,
        score_floor_factor=0.0,
        margin_floor=0.0,
    )
    out = ue.run().copy()
    out["dataset"] = dataset_name
    out["TrueMoA"] = out["CompoundID"].astype(str).map(true_moa_by_compound)
    out["correct"] = (out["PredictedMoA"].astype(str) == out["TrueMoA"].astype(str)).astype(int)
    out["error"] = 1 - out["correct"]
    return out


def summarize_flags(df, flag_col):
    d = df.copy()
    err = d["error"].astype(int).values
    flag = d[flag_col].astype(int).values

    tp = int(((flag == 1) & (err == 1)).sum())
    fp = int(((flag == 1) & (err == 0)).sum())
    tn = int(((flag == 0) & (err == 0)).sum())
    fn = int(((flag == 0) & (err == 1)).sum())

    n = len(d)
    coverage = float((flag == 0).mean()) if n else np.nan
    precision = float(tp / (tp + fp)) if (tp + fp) else np.nan
    recall = float(tp / (tp + fn)) if (tp + fn) else np.nan
    id_fpr = float(fp / (fp + tn)) if (fp + tn) else np.nan
    confident_acc = float(tn / (tn + fn)) if (tn + fn) else np.nan

    return {
        "n": n,
        "coverage": coverage,
        "precision": precision,
        "err_recall": recall,
        "id_fpr": id_fpr,
        "confident_acc": confident_acc,
    }


In [4]:
true_moa_by_compound = load_true_moa_by_compound()

dev_loo = evaluate_dev_root(LOO_ROOT, BASE_META_DIR, "LOO", true_moa_by_compound)
dev_loso = evaluate_dev_root(LOSO_ROOT, BASE_META_DIR, "LOSO", true_moa_by_compound)
dev_all = pd.concat([dev_loo, dev_loso], ignore_index=True)

print("Dev LOO shape:", dev_loo.shape)
print("Dev LOSO shape:", dev_loso.shape)
print("Base threshold:", float(base_artifact.get("meta_threshold", np.nan)))

base_thr = float(base_artifact.get("meta_threshold", 0.5))
dev_all["flag_base"] = (dev_all["p_error"] >= base_thr).astype(int)

loo_correct = dev_loo[dev_loo["correct"] == 1].copy()
if len(loo_correct) < 5:
    raise RuntimeError("Too few LOO-correct compounds for threshold recalibration")

new_thr = float(np.nanquantile(loo_correct["p_error"].values, 1.0 - ALPHA))
dev_all["flag_recal"] = (dev_all["p_error"] >= new_thr).astype(int)

print("Recalibrated threshold:", new_thr)

rows = []
for split_name, sub in [
    ("ALL", dev_all),
    ("LOO", dev_all[dev_all["dataset"] == "LOO"]),
    ("LOSO", dev_all[dev_all["dataset"] == "LOSO"]),
]:
    m_base = summarize_flags(sub, "flag_base")
    m_new = summarize_flags(sub, "flag_recal")
    rows.append({"split": split_name, "version": "base", **m_base})
    rows.append({"split": split_name, "version": "recalibrated", **m_new})

metrics_df = pd.DataFrame(rows)
metrics_df


Using existing pairwise CSVs in predictions/Uncertainty_experiment_semantic_proteomics/binary_loo_ablation/semantic_BioBERT_tabm_Text_Embedding+Proteomics
Using existing pairwise CSVs in predictions/Uncertainty_experiment_semantic_proteomics/binary_loso_ablation/semantic_BioBERT_tabm_Text_Embedding+Proteomics
Dev LOO shape: (51, 32)
Dev LOSO shape: (51, 32)
Base threshold: 0.4690518117171519
Recalibrated threshold: 0.27791674387824455


,split,version,n,coverage,precision,err_recall,id_fpr,confident_acc
0,ALL,base,102,0.990196,1.000000,0.019608,0.000000,0.504950
1,ALL,recalibrated,102,0.745098,0.576923,0.294118,0.215686,0.526316
2,LOO,base,51,1.000000,NaN,0.000000,0.000000,0.882353
3,LOO,recalibrated,51,0.862745,0.285714,0.333333,0.111111,0.909091
4,LOSO,base,51,0.980392,1.000000,0.022222,0.000000,0.120000
5,LOSO,recalibrated,51,0.627451,0.684211,0.288889,1.000000,0.000000


In [5]:
RECAL_META_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BASE_META_DIR / "meta_model.joblib", RECAL_META_DIR / "meta_model.joblib")

recal_artifact = dict(base_artifact)
recal_artifact["meta_threshold"] = float(new_thr)
recal_artifact["threshold_strategy"] = "recalibrated_on_semprot_dev_from_pairwise_csv"
recal_artifact["threshold_source"] = {
    "alpha": float(ALPHA),
    "loo_root": str(LOO_ROOT.resolve()),
    "loso_root": str(LOSO_ROOT.resolve()),
}

(RECAL_META_DIR / "meta_artifact.json").write_text(json.dumps(recal_artifact, indent=2))

print("Exported recalibrated meta dir:", RECAL_META_DIR)
print("Model:", RECAL_META_DIR / "meta_model.joblib")
print("Artifact:", RECAL_META_DIR / "meta_artifact.json")
print("meta_threshold:", recal_artifact["meta_threshold"])


Exported recalibrated meta dir: models/uncertainty_estimator_adapted
Model: models/uncertainty_estimator_adapted/meta_model.joblib
Artifact: models/uncertainty_estimator_adapted/meta_artifact.json
meta_threshold: 0.27791674387824455
